# MammoAI — обучение на VinDr-Mammo через Colab в VS Code
**Часть 1 — данные** (можно на обычном CPU-сервере, GPU-квота не тратится): 20 000 DICOM с PhysioNet → обрезка по груди → PNG 1536 px → пачки по 1000 в ваш **приватный** HF-датасет `mammoai-vindr` (~14 ГБ). Исходные DICOM (~300 ГБ) сохраняются только на ваш Google Drive, если `USE_DRIVE = True`. При обрыве просто запустите снова — продолжит с места остановки.

**Часть 2 — обучение** (нужен GPU): данные берутся из HF за минуты, PhysioNet уже не нужен. ~6 ч на T4, каждая модель сразу сохраняется в HF-репозиторий `mammoai-weights`.

**Run All** — выполнит обе части (часть 2 остановится с понятной ошибкой, если сервер без GPU).

In [ ]:
import os, subprocess, torch
from getpass import getpass
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'нет (подходит только для части 1)')
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass('Hugging Face write-токен: ')
os.environ['HF_TOKEN'] = HF_TOKEN
if not os.environ.get('PHYSIONET_USER'):
    # Only needed while some VinDr shards are not in your HF dataset yet.
    os.environ['PHYSIONET_USER'] = input('PhysioNet логин (Enter — если данные уже в HF): ').strip()
    if os.environ['PHYSIONET_USER']:
        os.environ['PHYSIONET_PASS'] = getpass('PhysioNet пароль: ')
HF_REPO = 'mammoai-weights'  # приватный репозиторий весов <ваш_логин>/mammoai-weights
SCREENING_EPOCHS = 12
CLASSIFIER_EPOCHS = 30
DETECTOR_EPOCHS = 40
RETRAIN = False  # True — обучать заново, даже если модель уже есть на этом сервере
USE_DRIVE = True  # True — сначала сырые DICOM на Google Drive через wget (1 соединение), потом конвертация оттуда
DRIVE_DIR = '/content/drive/MyDrive/vindr-mammo'  # папка с сырыми данными VinDr на Drive

import re, time
from IPython.display import display, HTML

ANSI = re.compile(r'\x1b\[[0-9;?]*[A-Za-z]')
BAR_KEY = re.compile(r'^\s*([A-Za-z][A-Za-z ()._-]*?)\s*[:(]')

def run(cmd):
    # Run a shell command, stream its output with live progress bars, stop the notebook if it fails.
    print('$', cmd, flush=True)
    env = dict(os.environ, PYTHONUNBUFFERED='1', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
    bars, handle, last = {}, None, 0.0

    def show(force=False):
        nonlocal handle, last
        if not bars or (not force and time.monotonic() - last < 1.0):
            return
        last = time.monotonic()
        text = '\n'.join(list(bars.values())[-4:])
        html = HTML('<pre style="margin:0">' + text.replace('&', '&amp;').replace('<', '&lt;') + '</pre>')
        if handle is None:
            handle = display(html, display_id=True)
        else:
            handle.update(html)

    buf = ''
    fd = p.stdout.fileno()
    while chunk := os.read(fd, 4096):
        buf += ANSI.sub('', chunk.decode(errors='replace'))
        while (i := min((buf.find(c) for c in '\r\n' if c in buf), default=-1)) >= 0:
            piece, sep, buf = buf[:i], buf[i], buf[i + 1:]
            if not piece.strip():
                continue
            if sep == '\n':
                print(piece, flush=True)
            else:  # progress-bar redraw: keep the latest state per bar
                m = BAR_KEY.match(piece)
                bars[m.group(1) if m else 'progress'] = piece.strip()
                show()
    show(force=True)
    if p.wait():
        raise RuntimeError(f'команда завершилась с ошибкой ({p.returncode}): {cmd}')

def upload(name):
    # Save trained weights to the private HF repo right away, so they survive a server disconnect.
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    repo = f"{api.whoami()['name']}/{HF_REPO}"
    api.create_repo(repo, private=True, exist_ok=True)
    api.upload_file(path_or_fileobj=f'models/{name}', path_in_repo=name, repo_id=repo)
    print(f'{name} → huggingface.co/{repo}')

In [ ]:
# Распаковка кода проекта (встроен в ноутбук)
import base64, io, zipfile
CODE = "..."
os.makedirs('/content/mammo', exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE))).extractall('/content/mammo')
os.chdir('/content/mammo')
print(sorted(os.listdir('.')))

In [ ]:
run('pip install -q timm ultralytics openpyxl pydicom pylibjpeg pylibjpeg-openjpeg')  # torch с CUDA уже есть

## Часть 1. Данные VinDr-Mammo
Печатает скорость (MB/s) и оставшееся время. Готовые пачки берутся из HF, недостающие делаются из DICOM — с Google Drive (`USE_DRIVE = True`) или напрямую с PhysioNet.

### Сырые DICOM на Google Drive (если `USE_DRIVE = True`)
Много параллельных соединений PhysioNet режет (таймауты подключения с IP Colab), поэтому датасет качается `wget` в **одно** соединение прямо на Google Drive, а конвертация потом идёт с Drive.

- Полный датасет — **≈ 300+ ГБ DICOM**: на Google Drive нужно столько свободного места (платный тариф Google One).
- Одной сессии Colab может не хватить — просто запустите ячейку снова, `wget` докачает с места остановки.
- В конце печатается, сколько `.dicom` уже на Drive из ожидаемых 20 000, и общий размер.

Сначала быстрая проверка `vindr_fetch.py status`: если все пачки уже в HF (`mammoai-vindr`), `wget` не запускается. Если данные уже в HF — Drive вообще не нужен, можно поставить `USE_DRIVE = False`.

In [ ]:
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        raise RuntimeError('Google Drive недоступен в этом окружении — поставьте USE_DRIVE = False '
                           'в первой ячейке') from e
    os.makedirs(DRIVE_DIR, exist_ok=True)
    # Cheap check: are any shards missing both here and in the HF dataset? (exit 0: none, 10: some)
    st = subprocess.run(['python', 'scripts/vindr_fetch.py', 'status', '--source', DRIVE_DIR],
                        capture_output=True, text=True)
    print(st.stdout + st.stderr)
    if st.returncode == 0:
        print('все пачки уже есть в HF — скачивание DICOM на Drive не нужно')
    elif st.returncode != 10:
        raise RuntimeError(f'не удалось проверить, какие пачки уже есть в HF (код {st.returncode}): '
                           'проверьте HF_TOKEN и сеть')
    elif os.environ.get('PHYSIONET_USER'):
        run(f'python scripts/vindr_to_drive.py --dest "{DRIVE_DIR}"')
    elif not os.path.exists(os.path.join(DRIVE_DIR, 'breast-level_annotations.csv')):
        print('ВНИМАНИЕ: части пачек нет в HF, а логин PhysioNet не задан и на Drive нет '
              'breast-level_annotations.csv — следующая ячейка не сможет сделать недостающие пачки. '
              'Перезапустите первую ячейку и введите логин PhysioNet.')
    else:
        print('логин PhysioNet не задан — wget пропущен, конвертация из того, что уже на Drive')
else:
    print('USE_DRIVE = False — DICOM будут качаться с PhysioNet напрямую в следующей ячейке')

: 

In [ ]:
if USE_DRIVE:
    run(f'python scripts/vindr_fetch.py download --source "{DRIVE_DIR}" --workers 8')
else:
    run('python scripts/vindr_fetch.py download --workers 4')  # больше соединений — PhysioNet режет IP
run('python -m mammo.labels')

## Часть 2. Обучение (GPU)
### BI-RADS всей груди 1–5 (~1.5 ч)

In [ ]:
assert torch.cuda.is_available(), 'Для обучения нужен GPU-сервер: Select Kernel → Colab → New Colab Server → GPU'
run('python -m mammo.prepare_yolo')

def train_once(name, cmd):
    # Skip a model already trained on this server (e.g. after a restart), then save it to HF.
    if os.path.exists(f'models/{name}') and not RETRAIN:
        print(f'models/{name} уже есть — пропускаю')
    else:
        run(cmd)
    upload(name)

train_once('screening.pt', f'python -m train.train_screening --epochs {SCREENING_EPOCHS} --batch 12 --workers 2 --device cuda')

### Классификатор очагов: BI-RADS находки + подозрение на рак (~0.5 ч)

In [ ]:
train_once('classifier.pt', f'python -m train.train_classifier --epochs {CLASSIFIER_EPOCHS} --batch 16 --workers 2 --device cuda')

### Детектор очагов YOLO11s, 1280 px (~3.5 ч)

In [ ]:
train_once('detector.pt', f'python -m train.train_detector --epochs {DETECTOR_EPOCHS} --batch 8 --device 0')

### Оценка на официальном test VinDr (1000 исследований)
Главное: `auc_1_2_vs_4_5`, чувствительность/специфичность, `false boxes per image` на здоровых, mAP.

Локально потом: `.venv/bin/hf download <логин>/mammoai-weights --local-dir models`

In [ ]:
run('python -m train.evaluate --device cuda')